In [0]:
%run "/Workspace/Denis_Databricks/Denis_DBx/denis_transformation/Generic"

In [0]:
%run "/Workspace/Denis_Databricks/Denis_DBx/denis_transformation/Connectors"

In [0]:
from pyspark.sql import functions as F

In [0]:
adls_connect()

In [0]:
list_silver_files()

In [0]:
cat_df = read_silver_file_csv("categories_S")
geo_df = read_silver_file_csv("geography_S")
prd_df = read_silver_file_csv("product_S")
salesRep_df = read_silver_file_csv("salesRep_S")
sales_df = read_silver_file_csv("sales_S")
subcat_df = read_silver_file_csv("subCategories_S")

In [0]:
display(cat_df.limit(5))
display(subcat_df.limit(5))
display(geo_df.limit(5))
display(prd_df.limit(5))
display(salesRep_df.limit(5))
display(sales_df.limit(5))

### Transformation

In [0]:
#Calculate Total Revenue in Sales table, using the Product’s Retail Price, and multiplying it by the Units.
sales_df = sales_df.join(prd_df, on="ProductID", how="inner")

# 2. Multiply Retail Price by Units to calculate Total Revenue
# Replace "RetailPrice" and "Units" with your exact column names
sales_df = sales_df.withColumn(
    "TotalRevenue", 
    F.round(F.col("RetailPrice") * F.col("Units"), 2)
)

In [0]:
display(sales_df.limit(5))

In [0]:
# Calculate Total Cost in Sales table, using the Product’s Standard Cost, and multiplying it by the Units.
sales_df = sales_df.withColumn("TotalCost",F.round(F.col("StandardCost") * F.col("Units"), 2))

In [0]:
display(sales_df.limit(5))

In [0]:
#Calculate Gross Profit in Sales: Total Revenue – Total Cost
sales_df = sales_df.withColumn("GrossProfit",F.round(F.col("TotalRevenue") - F.col("TotalCost"), 2))

In [0]:
display(sales_df.limit(5))

### Writing to Silver layer

In [0]:
write_to_gold(sales_df, "Denis_G.csv")